# MNIST 10-Seed Thesis Results

Runs the fixed MNIST configuration across 10 seeds, computes seed-averaged curves, checkpoint diagnostics, and a mean $\pm$ std table. No hyperparameter search is performed.


In [ ]:
import itertools
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision.datasets import MNIST

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
print(f"device={device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


METHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]
METHOD_LABELS = {
    "bp": "BP",
    "np": "IS-NP",
    "np_fan_in": "Fan-in NP",
    "np_fixed": "Vanilla NP",
    "wp": "WP",
}
METHOD_COLORS = {
    "bp": "C0",
    "np": "C1",
    "np_fan_in": "C4",
    "np_fixed": "C3",
    "wp": "C2",
}


class MLP(nn.Module):
    def __init__(self, dimensions, activation=torch.relu, output_activation=None, require_grad=False):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(dimensions[i], dimensions[i + 1], bias=True) for i in range(len(dimensions) - 1)]
        )
        self.activation = activation
        self.output_activation = output_activation
        if not require_grad:
            for p in self.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        final_layer = len(self.layers) - 1
        h = x
        for i, layer in enumerate(self.layers):
            u = layer(h)
            if i == final_layer:
                h = self.output_activation(u) if self.output_activation else u
            else:
                h = self.activation(u)
        return h

    def forward_weight_perturb(self, x, sigma):
        batch_size = x.shape[0]
        xs = [x]
        ys = []
        noises = []
        h = x
        for i, layer in enumerate(self.layers):
            w = layer.weight
            b = layer.bias
            eps_w = torch.randn(batch_size, *w.shape, device=w.device, dtype=w.dtype) * sigma
            eps_b = torch.randn(batch_size, *b.shape, device=b.device, dtype=b.dtype) * sigma
            w_used = w.unsqueeze(0) + eps_w
            b_used = b.unsqueeze(0) + eps_b
            u = torch.bmm(h.unsqueeze(1), w_used.transpose(1, 2)).squeeze(1) + b_used
            final_layer = len(self.layers) - 1
            if i == final_layer:
                h = self.output_activation(u) if self.output_activation else u
                ys.append(h)
            else:
                h = self.activation(u)
                ys.append(h)
                xs.append(h)
            noises.append((eps_w, eps_b))
        return ys, xs, noises

    def forward_node_perturb(self, x, sigma):
        acts = [x]
        noises = []
        noise_scales = []
        last = len(self.layers) - 1
        a_noisy = x
        for i, layer in enumerate(self.layers):
            x_in = a_noisy
            z_noisy = layer(a_noisy)
            eps = torch.randn_like(z_noisy)
            noise_scale = sigma * torch.sqrt(1.0 + x_in.pow(2).sum(dim=1, keepdim=True))
            noises.append(eps)
            noise_scales.append(noise_scale)
            z_noisy = z_noisy + noise_scale * eps
            if i == last:
                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy
            else:
                a_noisy = self.activation(z_noisy)
            acts.append(a_noisy)
        return acts, noises, noise_scales, a_noisy

    def forward_node_perturb_fixed_sigma(self, x, sigma):
        acts = [x]
        noises = []
        noise_scales = []
        last = len(self.layers) - 1
        a_noisy = x
        for i, layer in enumerate(self.layers):
            z_noisy = layer(a_noisy)
            eps = torch.randn_like(z_noisy)
            noise_scale = torch.full_like(z_noisy, sigma)
            noises.append(eps)
            noise_scales.append(noise_scale)
            z_noisy = z_noisy + noise_scale * eps
            if i == last:
                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy
            else:
                a_noisy = self.activation(z_noisy)
            acts.append(a_noisy)
        return acts, noises, noise_scales, a_noisy

    def forward_node_perturb_fan_in_scaled(self, x, sigma):
        acts = [x]
        noises = []
        noise_scales = []
        last = len(self.layers) - 1
        a_noisy = x
        for i, layer in enumerate(self.layers):
            z_noisy = layer(a_noisy)
            eps = torch.randn_like(z_noisy)
            fan_in_with_bias = layer.in_features + 1
            noise_scale_value = sigma * math.sqrt(fan_in_with_bias)
            noise_scale = torch.full_like(z_noisy, noise_scale_value)
            noises.append(eps)
            noise_scales.append(noise_scale)
            z_noisy = z_noisy + noise_scale * eps
            if i == last:
                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy
            else:
                a_noisy = self.activation(z_noisy)
            acts.append(a_noisy)
        return acts, noises, noise_scales, a_noisy


def _mean_loss_per_sample(prediction, target):
    loss = F.mse_loss(prediction, target, reduction="none")
    if loss.dim() > 1:
        loss = loss.mean(dim=1)
    return loss.view(-1)


def _centered_reward_signal(loss_per_sample):
    reward = -loss_per_sample
    return reward - reward.mean()


def _flatten_parameter_tensors(weight_tensors, bias_tensors):
    pieces = []
    for weight_tensor, bias_tensor in zip(weight_tensors, bias_tensors):
        pieces.append(weight_tensor.reshape(-1))
        pieces.append(bias_tensor.reshape(-1))
    return torch.cat(pieces)


def node_perturbation_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):
    model.train()
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(X, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    with torch.no_grad():
        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):
            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)
            raw_bias_update = scaled_noise.mean(dim=0)
            raw_weight_updates.append(raw_weight_update)
            raw_bias_updates.append(raw_bias_update)
            layer.weight += eta * raw_weight_update
            layer.bias += eta * raw_bias_update
    mean_loss = loss_per_sample.mean().item()
    if return_unscaled_parameter_update_vector:
        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)
    return mean_loss


def node_perturbation_step_fixed_sigma(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):
    model.train()
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(X, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    with torch.no_grad():
        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):
            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)
            raw_bias_update = scaled_noise.mean(dim=0)
            raw_weight_updates.append(raw_weight_update)
            raw_bias_updates.append(raw_bias_update)
            layer.weight += eta * raw_weight_update
            layer.bias += eta * raw_bias_update
    mean_loss = loss_per_sample.mean().item()
    if return_unscaled_parameter_update_vector:
        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)
    return mean_loss


def node_perturbation_step_fan_in_scaled(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):
    model.train()
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(X, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    with torch.no_grad():
        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):
            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)
            raw_bias_update = scaled_noise.mean(dim=0)
            raw_weight_updates.append(raw_weight_update)
            raw_bias_updates.append(raw_bias_update)
            layer.weight += eta * raw_weight_update
            layer.bias += eta * raw_bias_update
    mean_loss = loss_per_sample.mean().item()
    if return_unscaled_parameter_update_vector:
        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)
    return mean_loss


def weight_perturb_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):
    model.train()
    layer_outputs, _, noises = model.forward_weight_perturb(X, sigma)
    prediction_noisy = layer_outputs[-1]
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    noise_scale = sigma ** 2 + 1e-12
    raw_weight_updates = []
    raw_bias_updates = []
    with torch.no_grad():
        for layer, (weight_noise, bias_noise) in zip(model.layers, noises):
            scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale
            scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale
            raw_weight_update = scaled_weight_noise.mean(dim=0)
            raw_bias_update = scaled_bias_noise.mean(dim=0)
            raw_weight_updates.append(raw_weight_update)
            raw_bias_updates.append(raw_bias_update)
            layer.weight += eta * raw_weight_update
            layer.bias += eta * raw_bias_update
    mean_loss = loss_per_sample.mean().item()
    if return_unscaled_parameter_update_vector:
        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)
    return mean_loss


def backprop_step(model, X, target, optimizer, loss_fn=F.mse_loss, return_unscaled_parameter_update_vector=False):
    model.train()
    for p in model.parameters():
        p.requires_grad_(True)
    optimizer.zero_grad()
    y_pred = model(X)
    loss = loss_fn(y_pred, target, reduction="mean")
    loss.backward()
    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]
    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]
    optimizer.step()
    if return_unscaled_parameter_update_vector:
        return loss.item(), -_flatten_parameter_tensors(weight_grads, bias_grads)
    return loss.item()


def cosine_similarity_safe(a, b, eps=1e-12):
    a_norm = torch.norm(a)
    b_norm = torch.norm(b)
    if a_norm.item() < eps or b_norm.item() < eps:
        return 0.0
    return float(torch.dot(a, b) / (a_norm * b_norm + eps))


def true_gradient_vector(model, xb, yb):
    requires_grad_state = [parameter.requires_grad for parameter in model.parameters()]
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    model.zero_grad(set_to_none=True)
    prediction = model(xb)
    loss = F.mse_loss(prediction, yb, reduction="mean")
    loss.backward()
    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]
    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]
    flat_grad = _flatten_parameter_tensors(weight_grads, bias_grads)
    model.zero_grad(set_to_none=True)
    for parameter, old_value in zip(model.parameters(), requires_grad_state):
        parameter.requires_grad_(old_value)
    return -flat_grad


def gradient_metrics(unscaled_parameter_update_vector, true_update):
    diff = unscaled_parameter_update_vector - true_update
    cosine = cosine_similarity_safe(unscaled_parameter_update_vector, true_update)
    squared_error = float(diff.pow(2).mean())
    true_update_norm = float(torch.norm(true_update))
    projection = float(torch.dot(unscaled_parameter_update_vector, true_update) / (true_update_norm + 1e-12))
    return cosine, squared_error, projection


def node_perturbation_gradient_estimate_vector(model, xb, yb, sigma):
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(xb, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):
        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))
        raw_bias_updates.append(scaled_noise.mean(dim=0))
    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)


def node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma):
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(xb, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):
        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))
        raw_bias_updates.append(scaled_noise.mean(dim=0))
    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)


def node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma):
    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(xb, sigma)
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    raw_weight_updates = []
    raw_bias_updates = []
    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):
        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)
        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))
        raw_bias_updates.append(scaled_noise.mean(dim=0))
    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)


def weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma):
    layer_outputs, _, noises = model.forward_weight_perturb(xb, sigma)
    prediction_noisy = layer_outputs[-1]
    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)
    scalar_signal = _centered_reward_signal(loss_per_sample)
    noise_scale = sigma ** 2 + 1e-12
    raw_weight_updates = []
    raw_bias_updates = []
    for weight_noise, bias_noise in noises:
        scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale
        scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale
        raw_weight_updates.append(scaled_weight_noise.mean(dim=0))
        raw_bias_updates.append(scaled_bias_noise.mean(dim=0))
    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)


def perturbation_gradient_estimate_vector(method, model, xb, yb, sigma):
    if method == "np":
        return node_perturbation_gradient_estimate_vector(model, xb, yb, sigma)
    if method == "np_fan_in":
        return node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma)
    if method == "np_fixed":
        return node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma)
    if method == "wp":
        return weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma)
    raise ValueError(f"Unknown perturbation method: {method}")


def variance_against_true_gradient(estimate_matrix, true_update):
    num_samples = estimate_matrix.shape[0]
    if num_samples == 0:
        return 0.0, 0.0

    squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)
    per_sample_variance = float(squared_errors.mean(dim=1).mean())
    mean_estimate = estimate_matrix.mean(dim=0)
    batch_variance = float((mean_estimate - true_update).pow(2).mean())
    return per_sample_variance, batch_variance


def model_weight_norm(model):
    total = torch.tensor(0.0, device=next(model.parameters()).device)
    for layer in model.layers:
        total = total + layer.weight.detach().pow(2).sum()
    return float(torch.sqrt(total))


def mean_normalized_layer_output_norms(model, loader, device):
    model.eval()
    summed_norms = [0.0 for _ in model.layers]
    total_examples = 0

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                xb, _, _ = batch
            else:
                xb, _ = batch
            xb = xb.to(device, non_blocking=True)
            h = xb
            batch_size = xb.size(0)

            for layer_idx, layer in enumerate(model.layers):
                h_for_norm = h if h.dim() > 1 else h.unsqueeze(1)
                normalized_norm = torch.norm(h_for_norm, dim=1) / math.sqrt(h_for_norm.shape[1] + 1)
                summed_norms[layer_idx] += normalized_norm.sum().item()
                u = layer(h)
                if layer_idx == len(model.layers) - 1:
                    h = model.output_activation(u) if model.output_activation else u
                else:
                    h = model.activation(u)

            total_examples += batch_size

    return [value / max(total_examples, 1) for value in summed_norms]


def _random_subset_indices(n, limit, seed):
    if limit is None or limit >= n:
        return torch.arange(n)
    g = torch.Generator().manual_seed(seed)
    return torch.randperm(n, generator=g)[:limit]


def load_mnist(
    train_limit=20000,
    test_limit=5000,
    train_eval_limit=4000,
    batch_size=128,
    eval_batch_size=1024,
    data_dir="./data",
    seed=0,
    mean_center_only=True,
    num_workers=0,
):
    train_dataset = MNIST(root=data_dir, train=True, download=True)
    test_dataset = MNIST(root=data_dir, train=False, download=True)

    x_train = train_dataset.data.float() / 255.0
    x_test = test_dataset.data.float() / 255.0
    train_labels = train_dataset.targets.long()
    test_labels = test_dataset.targets.long()

    train_idx = _random_subset_indices(len(x_train), train_limit, seed)
    test_idx = _random_subset_indices(len(x_test), test_limit, seed + 1)

    x_train = x_train[train_idx]
    train_labels = train_labels[train_idx]
    x_test = x_test[test_idx]
    test_labels = test_labels[test_idx]

    mean = x_train.mean()
    if mean_center_only:
        x_train = x_train - mean
        x_test = x_test - mean
    else:
        std = x_train.std().clamp_min(1e-6)
        x_train = (x_train - mean) / std
        x_test = (x_test - mean) / std

    x_train = x_train.view(x_train.size(0), -1)
    x_test = x_test.view(x_test.size(0), -1)

    y_train = F.one_hot(train_labels, num_classes=10).float()
    y_test = F.one_hot(test_labels, num_classes=10).float()

    train_tensor_dataset = TensorDataset(x_train, y_train, train_labels)
    test_tensor_dataset = TensorDataset(x_test, y_test, test_labels)

    if train_eval_limit is None or train_eval_limit >= len(train_tensor_dataset):
        train_eval_dataset = train_tensor_dataset
    else:
        eval_idx = _random_subset_indices(len(train_tensor_dataset), train_eval_limit, seed + 2)
        train_eval_dataset = Subset(train_tensor_dataset, eval_idx.tolist())

    pin_memory = device.type == "cuda"
    train_loader = DataLoader(train_tensor_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    train_eval_loader = DataLoader(train_eval_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_tensor_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    return {
        "train_loader": train_loader,
        "train_eval_loader": train_eval_loader,
        "test_loader": test_loader,
        "train_size": len(train_tensor_dataset),
        "train_eval_size": len(train_eval_dataset),
        "test_size": len(test_tensor_dataset),
        "x_train": x_train,
        "y_train": y_train,
        "train_labels": train_labels,
        "x_test": x_test,
        "y_test": y_test,
        "test_labels": test_labels,
    }


def evaluate_model(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for xb, yb, labels in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(xb)
            loss = F.mse_loss(logits, yb, reduction="mean")
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += xb.size(0)
    mean_loss = total_loss / max(total_examples, 1)
    accuracy = total_correct / max(total_examples, 1)
    return mean_loss, accuracy


def make_model(dimensions, require_grad, device):
    return MLP(dimensions, activation=torch.relu, output_activation=None, require_grad=require_grad).to(device)


def train_one_run(
    method,
    run_config,
    data,
    dimensions,
    epochs,
    seed,
    device,
    print_every_epoch=1,
    divergence_loss_threshold=5.0,
):
    set_seed(seed)

    base_model = make_model(dimensions, require_grad=True, device=device)
    base_state = {name: tensor.detach().clone() for name, tensor in base_model.state_dict().items()}

    require_grad = method == "bp"
    model = make_model(dimensions, require_grad=require_grad, device=device)
    model.load_state_dict(base_state)

    optimizer = None
    if method == "bp":
        optimizer = torch.optim.SGD(model.parameters(), lr=run_config["lr"])

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "weight_norm": [],
        "layer_output_norms": [],
    }

    diverged = False
    best_test_acc = -float("inf")
    best_test_loss = float("inf")
    start_time = time.time()

    train_loader = data["train_loader"]
    train_eval_loader = data["train_eval_loader"]
    test_loader = data["test_loader"]

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb, _ in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            if method == "bp":
                backprop_step(model, xb, yb, optimizer=optimizer)
            elif method == "np":
                node_perturbation_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])
            elif method == "np_fan_in":
                node_perturbation_step_fan_in_scaled(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])
            elif method == "np_fixed":
                node_perturbation_step_fixed_sigma(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])
            elif method == "wp":
                weight_perturb_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])
            else:
                raise ValueError(f"Unknown method: {method}")

        train_loss, train_acc = evaluate_model(model, train_eval_loader, device)
        test_loss, test_acc = evaluate_model(model, test_loader, device)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["weight_norm"].append(model_weight_norm(model))
        history["layer_output_norms"].append(mean_normalized_layer_output_norms(model, train_eval_loader, device))

        best_test_acc = max(best_test_acc, test_acc)
        best_test_loss = min(best_test_loss, test_loss)

        if (not math.isfinite(train_loss)) or (not math.isfinite(test_loss)) or test_loss > divergence_loss_threshold:
            diverged = True
            print(
                f"    diverged at epoch {epoch}/{epochs} | "
                f"train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"
            )
            break

        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs:
            sigma_str = f", sigma={run_config['sigma']:.4g}" if "sigma" in run_config else ""
            print(
                f"    epoch {epoch:2d}/{epochs} | {method} | lr={run_config['lr']:.4g}{sigma_str} | "
                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"
            )

    duration_sec = time.time() - start_time
    state_dict_cpu = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    result = {
        "method": method,
        "lr": float(run_config["lr"]),
        "sigma": float(run_config["sigma"]) if "sigma" in run_config else np.nan,
        "epochs_completed": len(history["epoch"]),
        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,
        "final_train_acc": history["train_acc"][-1] if history["train_acc"] else np.nan,
        "final_test_loss": history["test_loss"][-1] if history["test_loss"] else np.nan,
        "final_test_acc": history["test_acc"][-1] if history["test_acc"] else np.nan,
        "best_test_loss": best_test_loss,
        "best_test_acc": best_test_acc,
        "diverged": diverged,
        "duration_sec": duration_sec,
        "history": history,
        "state_dict": state_dict_cpu,
    }
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result


def train_backprop_checkpoint_states(data, dimensions, epochs, seed, device, lr, checkpoint_epochs, print_every_epoch=100):
    checkpoint_epochs = sorted(set(int(epoch) for epoch in checkpoint_epochs))
    set_seed(seed)
    model = make_model(dimensions, require_grad=True, device=device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    checkpoint_states = {}

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb, _ in data["train_loader"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            backprop_step(model, xb, yb, optimizer=optimizer)

        if epoch in checkpoint_epochs:
            checkpoint_states[epoch] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs or epoch in checkpoint_epochs:
            train_loss, train_acc = evaluate_model(model, data["train_eval_loader"], device)
            test_loss, test_acc = evaluate_model(model, data["test_loader"], device)
            print(
                f"    bp checkpoint training epoch {epoch:4d}/{epochs} | "
                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"
            )

    return checkpoint_states


def plot_layer_output_norm_histories(results_by_method):
    first_result = next((results_by_method[method] for method in METHODS if method in results_by_method), None)
    if first_result is None:
        return

    layer_output_history = first_result["history"].get("layer_output_norms", [])
    if len(layer_output_history) == 0:
        return

    num_layers = len(layer_output_history[0])
    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)
    axes = axes.flatten()

    for layer_idx in range(num_layers):
        fan_in = int(first_result["state_dict"][f"layers.{layer_idx}.weight"].shape[1])
        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"
        axis = axes[layer_idx]
        for method in METHODS:
            if method not in results_by_method:
                continue
            history = results_by_method[method]["history"]
            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms", [])]
            if len(values) == 0:
                continue
            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])
        axis.set_title(f"{layer_label} norm / sqrt({fan_in} + 1)")
        axis.set_ylabel("Average norm")
        axis.legend(ncol=2)

    axes[-1].set_xlabel("Epoch")
    fig.tight_layout()
    plt.show()


def layer_parameter_slices(model):
    layer_slices = []
    start = 0
    for layer_idx, layer in enumerate(model.layers):
        layer_parameter_count = layer.weight.numel() + layer.bias.numel()
        layer_slices.append((layer_idx, slice(start, start + layer_parameter_count), f"Layer {layer_idx + 1}"))
        start += layer_parameter_count
    return layer_slices


def analyze_frozen_backprop_estimators(
    checkpoint_states,
    data,
    dimensions,
    device,
    method_sigmas,
    num_perturbations=50,
    batch_size=128,
):
    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])
    analysis_loader = DataLoader(
        analysis_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    rows = []
    perturbation_methods = list(method_sigmas.keys())

    for checkpoint_epoch, state_dict in checkpoint_states.items():
        model = make_model(dimensions, require_grad=True, device=device)
        model.load_state_dict(state_dict)
        component_specs = [(-1, slice(None), "All layers")] + layer_parameter_slices(model)

        batch_metrics = {
            method: {
                layer_index: {
                    "component": "all" if layer_index == -1 else f"layer_{layer_index + 1}",
                    "component_label": component_label,
                    "avg_sample_cosine": [],
                    "mean_estimate_cosine": [],
                    "sample_variance": [],
                    "batch_variance": [],
                }
                for layer_index, _, component_label in component_specs
            }
            for method in perturbation_methods
        }

        for xb, yb in analysis_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            true_update = true_gradient_vector(model, xb, yb)

            for method in perturbation_methods:
                sigma = method_sigmas[method]
                estimates = [
                    perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)
                    for _ in range(num_perturbations)
                ]
                estimate_matrix = torch.stack(estimates, dim=0)
                mean_estimate = estimate_matrix.mean(dim=0)

                for layer_index, component_slice, _ in component_specs:
                    component_true_update = true_update[component_slice]
                    component_estimate_matrix = estimate_matrix[:, component_slice]
                    component_mean_estimate = mean_estimate[component_slice]

                    avg_sample_cosine = float(
                        np.mean(
                            [
                                cosine_similarity_safe(component_estimate, component_true_update)
                                for component_estimate in component_estimate_matrix
                            ]
                        )
                    )
                    mean_estimate_cosine = cosine_similarity_safe(component_mean_estimate, component_true_update)
                    sample_variance, batch_variance = variance_against_true_gradient(
                        component_estimate_matrix,
                        component_true_update,
                    )

                    metrics = batch_metrics[method][layer_index]
                    metrics["avg_sample_cosine"].append(avg_sample_cosine)
                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)
                    metrics["sample_variance"].append(sample_variance)
                    metrics["batch_variance"].append(batch_variance)

        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())
        for method in perturbation_methods:
            for layer_index, _, component_label in component_specs:
                metrics = batch_metrics[method][layer_index]
                rows.append({
                    "checkpoint_epoch": checkpoint_epoch,
                    "checkpoint_fraction": checkpoint_fraction,
                    "method": method,
                    "sigma": method_sigmas[method],
                    "component": metrics["component"],
                    "component_label": component_label,
                    "layer_index": layer_index,
                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),
                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),
                    "sample_variance": float(np.mean(metrics["sample_variance"])),
                    "batch_variance": float(np.mean(metrics["batch_variance"])),
                })

    return pd.DataFrame(rows)


def plot_frozen_estimator_statistics(stats_df):
    metrics = [
        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),
        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),
        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),
        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),
    ]
    perturbation_methods = [method for method in METHODS if method in {"np", "np_fan_in", "np_fixed", "wp"}]

    overall_df = stats_df[stats_df["layer_index"] == -1] if "layer_index" in stats_df.columns else stats_df
    fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4), sharex=True)

    for method in perturbation_methods:
        method_df = overall_df[overall_df["method"] == method].sort_values("checkpoint_epoch")
        if len(method_df) == 0:
            continue
        x = method_df["checkpoint_epoch"].to_numpy()
        for axis, (column, title, ylabel) in zip(axes, metrics):
            axis.plot(x, method_df[column].to_numpy(), marker="o", color=METHOD_COLORS[method], label=METHOD_LABELS[method])
            axis.set_title(f"All layers | {title}")
            axis.set_xlabel("Checkpoint epoch")
            axis.set_ylabel(ylabel)
            axis.legend()

    fig.tight_layout()
    plt.show()

    if "layer_index" not in stats_df.columns:
        return

    layer_indices = sorted(layer_index for layer_index in stats_df["layer_index"].unique() if layer_index >= 0)
    if len(layer_indices) == 0:
        return

    fig, axes = plt.subplots(
        len(layer_indices),
        len(metrics),
        figsize=(6 * len(metrics), 3.8 * len(layer_indices)),
        sharex=True,
        squeeze=False,
    )

    for row, layer_index in enumerate(layer_indices):
        layer_df = stats_df[stats_df["layer_index"] == layer_index]
        component_label = layer_df["component_label"].iloc[0]

        for col, (column, title, ylabel) in enumerate(metrics):
            axis = axes[row, col]
            for method in perturbation_methods:
                method_df = layer_df[layer_df["method"] == method].sort_values("checkpoint_epoch")
                if len(method_df) == 0:
                    continue
                axis.plot(
                    method_df["checkpoint_epoch"].to_numpy(),
                    method_df[column].to_numpy(),
                    marker="o",
                    color=METHOD_COLORS[method],
                    label=METHOD_LABELS[method],
                )
            axis.set_title(f"{component_label} | {title}")
            axis.set_xlabel("Checkpoint epoch")
            axis.set_ylabel(ylabel)
            if col == len(metrics) - 1:
                axis.legend(fontsize=8)

    fig.tight_layout()
    plt.show()


def analyze_frozen_backprop_sigma_grid(
    checkpoint_states,
    data,
    dimensions,
    device,
    method_sigma_grid,
    num_perturbations=50,
    batch_size=128,
):
    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])
    analysis_loader = DataLoader(
        analysis_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    rows = []
    methods = list(method_sigma_grid.keys())

    for checkpoint_epoch, state_dict in checkpoint_states.items():
        model = make_model(dimensions, require_grad=True, device=device)
        model.load_state_dict(state_dict)

        batch_metrics = {
            (method, sigma): {
                "avg_sample_cosine": [],
                "mean_estimate_cosine": [],
                "sample_variance": [],
                "batch_variance": [],
            }
            for method in methods
            for sigma in method_sigma_grid[method]
        }

        for batch in analysis_loader:
            if len(batch) == 3:
                xb, yb, _ = batch
            else:
                xb, yb = batch
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            true_update = true_gradient_vector(model, xb, yb)

            for method in methods:
                for sigma in method_sigma_grid[method]:
                    estimates = [
                        perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)
                        for _ in range(num_perturbations)
                    ]
                    estimate_matrix = torch.stack(estimates, dim=0)
                    mean_estimate = estimate_matrix.mean(dim=0)

                    avg_sample_cosine = float(
                        sum(cosine_similarity_safe(estimate, true_update) for estimate in estimates) / num_perturbations
                    )
                    mean_estimate_cosine = cosine_similarity_safe(mean_estimate, true_update)
                    sample_variance, batch_variance = variance_against_true_gradient(estimate_matrix, true_update)

                    metrics = batch_metrics[(method, sigma)]
                    metrics["avg_sample_cosine"].append(avg_sample_cosine)
                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)
                    metrics["sample_variance"].append(sample_variance)
                    metrics["batch_variance"].append(batch_variance)

        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())
        for method in methods:
            for sigma in method_sigma_grid[method]:
                metrics = batch_metrics[(method, sigma)]
                rows.append({
                    "checkpoint_epoch": checkpoint_epoch,
                    "checkpoint_fraction": checkpoint_fraction,
                    "method": method,
                    "sigma": float(sigma),
                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),
                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),
                    "sample_variance": float(np.mean(metrics["sample_variance"])),
                    "batch_variance": float(np.mean(metrics["batch_variance"])),
                })

    return pd.DataFrame(rows)


def plot_frozen_sigma_search_results(stats_df):
    metrics = [
        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),
        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),
        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),
        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),
    ]
    checkpoint_epochs = sorted(stats_df["checkpoint_epoch"].unique())

    for method in [method for method in METHODS if method in set(stats_df["method"])] :
        method_df = stats_df[stats_df["method"] == method]
        fig, axes = plt.subplots(
            len(checkpoint_epochs),
            len(metrics),
            figsize=(5 * len(metrics), 3.8 * len(checkpoint_epochs)),
            squeeze=False,
        )

        for row, checkpoint_epoch in enumerate(checkpoint_epochs):
            subset = method_df[method_df["checkpoint_epoch"] == checkpoint_epoch].sort_values("sigma")
            sigma_labels = [f"{sigma:.4g}" for sigma in subset["sigma"].to_numpy()]

            for col, (column, title, ylabel) in enumerate(metrics):
                axis = axes[row, col]
                axis.bar(sigma_labels, subset[column].to_numpy(), color=METHOD_COLORS[method])
                axis.set_title(f"epoch {checkpoint_epoch} | {title}")
                axis.set_xlabel("sigma")
                axis.set_ylabel(ylabel)
                axis.tick_params(axis="x", rotation=45)

        fig.suptitle(f"{METHOD_LABELS[method]} sigma search", y=1.02)
        fig.tight_layout()
        plt.show()


def build_grid(space):
    keys = list(space.keys())
    values = [space[key] for key in keys]
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]


def run_sweep(search_spaces, data, dimensions, sweep_epochs, seeds, device):
    results = []
    flat_records = []
    total_runs = sum(len(configs) * len(seeds) for configs in search_spaces.values())
    run_idx = 0

    print(f"Starting sweep with total_runs={total_runs}, device={device}")
    for method in METHODS:
        configs = search_spaces[method]
        for config in configs:
            for seed in seeds:
                run_idx += 1
                sigma_str = f", sigma={config['sigma']:.4g}" if "sigma" in config else ""
                print(f"\n=== Run {run_idx}/{total_runs} | {method} | lr={config['lr']:.4g}{sigma_str} | seed={seed} ===")
                result = train_one_run(
                    method=method,
                    run_config=config,
                    data=data,
                    dimensions=dimensions,
                    epochs=sweep_epochs,
                    seed=seed,
                    device=device,
                    print_every_epoch=1,
                )
                results.append(result)
                flat_record = {k: v for k, v in result.items() if k not in {"history", "state_dict"}}
                flat_record["seed"] = seed
                flat_records.append(flat_record)

    df = pd.DataFrame(flat_records)
    if len(df) > 0:
        df = df.sort_values(
            ["method", "final_test_acc", "best_test_acc", "final_test_loss"],
            ascending=[True, False, False, True],
        ).reset_index(drop=True)
    return results, df


def summarize_top_configs(df, top_k=5):
    if len(df) == 0:
        return df
    frames = []
    for method, group in df.groupby("method"):
        frames.append(group.head(top_k))
    return pd.concat(frames, ignore_index=True)


def select_best_configs(df):
    rows = []
    good = df[df["diverged"] == False]
    for method, group in good.groupby("method"):
        best = group.sort_values(
            ["final_test_acc", "best_test_acc", "final_test_loss"],
            ascending=[False, False, True],
        ).iloc[0]
        rows.append(best)
    return pd.DataFrame(rows).reset_index(drop=True)


def run_best_config_comparison(best_df, data, dimensions, final_epochs, seed, device):
    comparison_results = {}
    for row in best_df.itertuples(index=False):
        config = {"lr": float(row.lr)}
        if not pd.isna(row.sigma):
            config["sigma"] = float(row.sigma)
        print(
            f"\n### Final run | {row.method} | lr={config['lr']:.4g}" +
            (f", sigma={config['sigma']:.4g}" if 'sigma' in config else "")
        )
        comparison_results[row.method] = train_one_run(
            method=row.method,
            run_config=config,
            data=data,
            dimensions=dimensions,
            epochs=final_epochs,
            seed=seed,
            device=device,
            print_every_epoch=1,
        )
    return comparison_results


def results_table(results_by_method):
    rows = []
    for method in METHODS:
        if method not in results_by_method:
            continue
        result = results_by_method[method]
        rows.append(
            {
                "method": method,
                "final_train_loss": result["final_train_loss"],
                "final_train_acc": result["final_train_acc"],
                "final_test_loss": result["final_test_loss"],
                "final_test_acc": result["final_test_acc"],
                "best_test_loss": result["best_test_loss"],
                "best_test_acc": result["best_test_acc"],
                "diverged": result["diverged"],
                "duration_sec": result["duration_sec"],
            }
        )
    return pd.DataFrame(rows)


def plot_training_histories(results_by_method):
    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
    for method in METHODS:
        if method not in results_by_method:
            continue
        result = results_by_method[method]
        history = result["history"]
        axes[0].plot(history["epoch"], history["train_loss"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])
        axes[0].plot(history["epoch"], history["test_loss"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])
        axes[1].plot(history["epoch"], history["train_acc"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])
        axes[1].plot(history["epoch"], history["test_acc"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])
        axes[2].plot(history["epoch"], history["weight_norm"], label=METHOD_LABELS[method], color=METHOD_COLORS[method])
    axes[0].set_title("Loss")
    axes[0].set_ylabel("MSE")
    axes[0].legend(ncol=2)
    axes[1].set_title("Accuracy")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].legend(ncol=2)
    axes[2].set_title("Weight Norm")
    axes[2].set_ylabel("L2 norm")
    axes[2].set_xlabel("Epoch")
    axes[2].legend(ncol=2)
    fig.tight_layout()
    plt.show()


# Plot labels are overwritten below by the fixed task config cell.


In [ ]:
from pathlib import Path
import gc
import pickle
import shutil
import tempfile
from IPython.display import display

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_DIR / "mnist_10seed_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
DATA_OUT_DIR = OUTPUT_DIR / "data"
DATA_DIR = BASE_DIR / "data"
for directory in [FIGURE_DIR, TABLE_DIR, DATA_OUT_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
})

TASK_CONFIG = {
    "task_key": "mnist",
    "display_name": "MNIST",
    "task_type": "classification",
    "data_loader": "load_mnist",
    "dimensions": (28 * 28, 256, 128, 10),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "seeds": list(range(5)),
    "data_kwargs": {
        'train_limit': 20000,
        'test_limit': 5000,
        'train_eval_limit': 4000,
        'batch_size': 128,
        'eval_batch_size': 1024,
        'data_dir': str(DATA_DIR / "torchvision"),
        'seed': 0,
        'mean_center_only': True,
        'num_workers': 0,
    },
    "run_epochs": 20 * 20,
    "run_print_every_epoch": 25,
    "run_configs": {
        "bp": {"lr": 0.5},
        "np": {"lr": 0.12 * 0.425, "sigma": 0.0325},
        "np_fan_in": {"lr": 0.0325 * 0.375, "sigma": 0.00875},
        "np_fixed": {"lr": 0.0225 * 0.25, "sigma": 0.12},
        "wp": {"lr": 0.015 * 0.25, "sigma": 0.0275},
    },
    "analysis": {
        "epochs": 20,
        "bp_lr": 0.2,
        "num_perturbations": 100,
        "batch_size": 128,
        "checkpoint_epochs": [1, 10, 20],
        "method_sigmas": {
                "np": 0.0325,
                "np_fan_in": 0.00875,
                "np_fixed": 0.12,
                "wp": 0.0275,
        },
    },
}

# Thesis plot labels use the no-hyphen abbreviation for IS NP.
METHOD_LABELS = {
    "bp": "BP",
    "np": "IS NP",
    "np_fan_in": "Fan-in NP",
    "np_fixed": "Vanilla NP",
    "wp": "WP",
}
METHOD_COLORS = {
    "bp": "#1f77b4",
    "np": "#ff7f0e",
    "np_fan_in": "#9467bd",
    "np_fixed": "#d62728",
    "wp": "#2ca02c",
}
PERTURBATION_METHODS = ["np", "np_fan_in", "np_fixed", "wp"]
CONVERGENCE_FRACTION = 0.90
VARIANCE_COLUMN = "sample_variance"
COSINE_COLUMN = "mean_estimate_cosine"

print(f"Output directory: {OUTPUT_DIR}")
print(f"Device: {device}")
print(f"Seeds: {TASK_CONFIG['seeds']}")
print("Fixed run configs:")
for method, cfg in TASK_CONFIG["run_configs"].items():
    print(f"  {method}: {cfg}")
print("Diagnostic perturbation samples:", TASK_CONFIG["analysis"]["num_perturbations"])


In [ ]:

def load_task_data(config):
    loader = globals()[config["data_loader"]]
    return loader(**config["data_kwargs"])


def convergence_epoch_from_history(history, fraction=CONVERGENCE_FRACTION):
    epochs = history.get("epoch", [])
    losses = history.get("test_loss", [])
    pairs = [(int(e), float(l)) for e, l in zip(epochs, losses) if np.isfinite(float(l))]
    if not pairs:
        return np.nan
    initial_loss = pairs[0][1]
    best_loss = min(loss for _, loss in pairs)
    improvement = initial_loss - best_loss
    if improvement <= 1e-12:
        return min(epoch for epoch, loss in pairs if loss == best_loss)
    threshold = initial_loss - fraction * improvement
    for epoch, loss in pairs:
        if loss <= threshold:
            return epoch
    return pairs[-1][0]


def history_value(history, key, index, default=np.nan):
    values = history.get(key, [])
    if index < len(values):
        return float(values[index])
    return default


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_task_10seed(config):
    data = load_task_data(config)
    print(
        f"Dataset: train={data['train_size']}, train_eval={data.get('train_eval_size', data['train_size'])}, "
        f"test={data['test_size']}, dims={config['dimensions']}, batch={config['data_kwargs']['batch_size']}"
    )

    is_classification = config["task_type"] == "classification"
    history_rows = []
    summary_seed_rows = []
    diagnostic_dfs = []

    total_seeds = len(config["seeds"])
    for seed_index, seed in enumerate(config["seeds"], start=1):
        print(f"\n===== Seed {seed_index}/{total_seeds}: {seed} =====")

        for method_index, method in enumerate(config["methods"], start=1):
            run_config = config["run_configs"][method]
            sigma_str = f", sigma={run_config['sigma']:.4g}" if "sigma" in run_config else ""
            print(f"\n[{seed_index}/{total_seeds}] Training {METHOD_LABELS[method]} ({method_index}/{len(config['methods'])}) | lr={run_config['lr']:.4g}{sigma_str}")
            result = train_one_run(
                method=method,
                run_config=run_config,
                data=data,
                dimensions=config["dimensions"],
                epochs=config["run_epochs"],
                seed=seed,
                device=device,
                print_every_epoch=config["run_print_every_epoch"],
            )
            history = result["history"]
            for i, epoch in enumerate(history["epoch"]):
                row = {
                    "seed": seed,
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "epoch": int(epoch),
                    "train_loss": history_value(history, "train_loss", i),
                    "test_loss": history_value(history, "test_loss", i),
                }
                if is_classification:
                    row["train_score"] = history_value(history, "train_acc", i)
                    row["test_score"] = history_value(history, "test_acc", i)
                    row["train_acc"] = row["train_score"]
                    row["test_acc"] = row["test_score"]
                else:
                    row["train_score"] = history_value(history, "train_r2", i)
                    row["test_score"] = history_value(history, "test_r2", i)
                    row["train_r2"] = row["train_score"]
                    row["test_r2"] = row["test_score"]
                history_rows.append(row)

            train_score_key = "train_acc" if is_classification else "train_r2"
            test_score_key = "test_acc" if is_classification else "test_r2"
            summary_seed_rows.append({
                "seed": seed,
                "task": config["display_name"],
                "method": method,
                "method_label": METHOD_LABELS[method],
                "best_train_loss": float(np.nanmin(history["train_loss"])),
                "best_test_loss": float(np.nanmin(history["test_loss"])),
                "best_train_score": float(np.nanmax(history[train_score_key])),
                "best_test_score": float(np.nanmax(history[test_score_key])),
                "score_metric": "Accuracy" if is_classification else "R2",
                "convergence_epoch": convergence_epoch_from_history(history),
                "epochs_completed": int(result["epochs_completed"]),
                "diverged": bool(result["diverged"]),
            })
            del result
            clear_gpu_memory()

        analysis = config["analysis"]
        print(
            f"\n[{seed_index}/{total_seeds}] Diagnostics | BP checkpoints={analysis['checkpoint_epochs']} | "
            f"perturbations={analysis['num_perturbations']}"
        )
        checkpoint_states = train_backprop_checkpoint_states(
            data=data,
            dimensions=config["dimensions"],
            epochs=analysis["epochs"],
            seed=seed,
            device=device,
            lr=analysis["bp_lr"],
            checkpoint_epochs=analysis["checkpoint_epochs"],
            print_every_epoch=max(1, analysis["epochs"] // 4),
        )
        diagnostic_df = analyze_frozen_backprop_estimators(
            checkpoint_states=checkpoint_states,
            data=data,
            dimensions=config["dimensions"],
            device=device,
            method_sigmas=analysis["method_sigmas"],
            num_perturbations=analysis["num_perturbations"],
            batch_size=analysis["batch_size"],
        )
        diagnostic_df.insert(0, "seed", seed)
        diagnostic_dfs.append(diagnostic_df)
        del checkpoint_states, diagnostic_df
        clear_gpu_memory()

    history_df = pd.DataFrame(history_rows)
    summary_seed_df = pd.DataFrame(summary_seed_rows)
    diagnostics_df = pd.concat(diagnostic_dfs, ignore_index=True)
    return data, history_df, summary_seed_df, diagnostics_df


data, history_df, summary_seed_df, diagnostics_df = run_task_10seed(TASK_CONFIG)
print(f"\nFinished 10-seed {TASK_CONFIG['display_name']} evaluation.")
display(summary_seed_df.head())
display(diagnostics_df.head())


In [ ]:

def aggregate_history(history_df):
    metric_cols = ["train_loss", "test_loss", "train_score", "test_score"]
    return (
        history_df.groupby(["method", "method_label", "epoch"], as_index=False)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )


def aggregate_diagnostics(diagnostics_df):
    all_layers = diagnostics_df[diagnostics_df["component"] == "all"].copy()
    checkpoint_seed = (
        all_layers.groupby(["seed", "checkpoint_epoch", "method"], as_index=False)
        .agg(
            cosine=(COSINE_COLUMN, "mean"),
            variance=(VARIANCE_COLUMN, "mean"),
            sigma=("sigma", "mean"),
        )
    )
    checkpoint_summary = (
        checkpoint_seed.groupby(["checkpoint_epoch", "method"], as_index=False)
        .agg(
            cosine_mean=("cosine", "mean"),
            cosine_std=("cosine", "std"),
            variance_mean=("variance", "mean"),
            variance_std=("variance", "std"),
            sigma=("sigma", "mean"),
        )
    )
    seed_over_checkpoints = (
        checkpoint_seed.groupby(["seed", "method"], as_index=False)
        .agg(
            cosine=("cosine", "mean"),
            variance=("variance", "mean"),
            sigma=("sigma", "mean"),
        )
    )
    overall_summary = (
        seed_over_checkpoints.groupby("method", as_index=False)
        .agg(
            cosine_mean=("cosine", "mean"),
            cosine_std=("cosine", "std"),
            variance_mean=("variance", "mean"),
            variance_std=("variance", "std"),
            sigma=("sigma", "mean"),
        )
    )
    return checkpoint_seed, checkpoint_summary, seed_over_checkpoints, overall_summary


def aggregate_table(summary_seed_df, seed_over_checkpoints):
    table_seed = summary_seed_df.merge(seed_over_checkpoints[["seed", "method", "cosine", "variance"]], on=["seed", "method"], how="left")
    table_seed.loc[table_seed["method"] == "bp", "cosine"] = 1.0
    table_seed.loc[table_seed["method"] == "bp", "variance"] = 0.0
    cols = ["best_train_loss", "best_test_loss", "best_train_score", "best_test_score", "convergence_epoch", "cosine", "variance"]
    table_summary = (
        table_seed.groupby(["task", "method", "method_label", "score_metric"], as_index=False)
        .agg(
            best_train_loss_mean=("best_train_loss", "mean"),
            best_train_loss_std=("best_train_loss", "std"),
            best_test_loss_mean=("best_test_loss", "mean"),
            best_test_loss_std=("best_test_loss", "std"),
            best_train_score_mean=("best_train_score", "mean"),
            best_train_score_std=("best_train_score", "std"),
            best_test_score_mean=("best_test_score", "mean"),
            best_test_score_std=("best_test_score", "std"),
            convergence_epoch_mean=("convergence_epoch", "mean"),
            convergence_epoch_std=("convergence_epoch", "std"),
            cosine_mean=("cosine", "mean"),
            cosine_std=("cosine", "std"),
            variance_mean=("variance", "mean"),
            variance_std=("variance", "std"),
        )
    )
    return table_seed, table_summary


history_summary_df = aggregate_history(history_df)
checkpoint_seed_df, checkpoint_summary_df, diagnostic_seed_summary_df, diagnostic_summary_df = aggregate_diagnostics(diagnostics_df)
table_seed_df, table_summary_df = aggregate_table(summary_seed_df, diagnostic_seed_summary_df)

print("History rows:", len(history_df))
print("Diagnostics rows:", len(diagnostics_df))
print("Table seed rows:", len(table_seed_df))
display(table_summary_df)


In [ ]:

def save_pdf(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return path


def add_curve_legend(ax, methods):
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=METHOD_COLORS[m], lw=1.5, label=METHOD_LABELS[m]) for m in methods]
    handles.extend([
        Line2D([0], [0], color="black", lw=1.5, linestyle="-", label="Training"),
        Line2D([0], [0], color="black", lw=1.5, linestyle="--", label="Test"),
    ])
    ax.legend(handles=handles, ncol=2, frameon=True, framealpha=0.95)


def plot_loss_curves_10seed(history_df, filename=None):
    filename = filename or f"{TASK_CONFIG['task_key']}_loss_10seed.pdf"
    grouped = history_df.groupby(["method", "epoch"], as_index=False).agg(train_loss=("train_loss", "mean"), test_loss=("test_loss", "mean"))
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    for method in TASK_CONFIG["methods"]:
        method_df = grouped[grouped["method"] == method].sort_values("epoch")
        ax.plot(method_df["epoch"], method_df["train_loss"], color=METHOD_COLORS[method], lw=1.15, linestyle="-")
        ax.plot(method_df["epoch"], method_df["test_loss"], color=METHOD_COLORS[method], lw=1.15, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    add_curve_legend(ax, TASK_CONFIG["methods"])
    fig.tight_layout()
    return save_pdf(fig, filename)


def plot_score_curves_10seed(history_df, filename=None):
    filename = filename or f"{TASK_CONFIG['task_key']}_accuracy_10seed.pdf"
    grouped = history_df.groupby(["method", "epoch"], as_index=False).agg(train_score=("train_score", "mean"), test_score=("test_score", "mean"))
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    for method in TASK_CONFIG["methods"]:
        method_df = grouped[grouped["method"] == method].sort_values("epoch")
        ax.plot(method_df["epoch"], method_df["train_score"], color=METHOD_COLORS[method], lw=1.15, linestyle="-")
        ax.plot(method_df["epoch"], method_df["test_score"], color=METHOD_COLORS[method], lw=1.15, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy" if TASK_CONFIG["task_type"] == "classification" else "R2")
    if TASK_CONFIG["task_type"] == "classification":
        ax.set_ylim(0.0, 1.0)
    add_curve_legend(ax, TASK_CONFIG["methods"])
    fig.tight_layout()
    return save_pdf(fig, filename)


def cosine_axis_limits(values, padding_fraction=0.15):
    finite_values = np.asarray([float(v) for v in values if pd.notna(v) and np.isfinite(float(v))])
    if finite_values.size == 0:
        return 0.0, 1.0
    min_value = float(finite_values.min())
    max_value = float(finite_values.max())
    if min_value >= 0.0:
        upper = max_value * (1.0 + padding_fraction) if max_value > 0.0 else 0.05
        return 0.0, min(1.0, upper)
    value_range = max_value - min_value
    padding = value_range * padding_fraction if value_range > 0.0 else 0.05
    return max(-1.0, min_value - padding), min(1.0, max_value + padding)


def plot_diagnostic_bar(summary_df, value_mean_col, ylabel, filename, log_scale=False):
    plot_df = summary_df.set_index("method").loc[PERTURBATION_METHODS].reset_index()
    values = plot_df[value_mean_col].astype(float).to_numpy()
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    x = np.arange(len(plot_df))
    for idx, row in plot_df.iterrows():
        method = row["method"]
        ax.bar(x[idx], row[value_mean_col], color=METHOD_COLORS[method], width=0.72, label=METHOD_LABELS[method])
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[m] for m in plot_df["method"]], rotation=20, ha="right")
    ax.set_ylabel(f"{ylabel} (log scale)" if log_scale else ylabel)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, framealpha=0.95)
    if log_scale:
        positive = values[values > 0]
        ax.set_yscale("log")
        ax.set_ylim(float(positive.min()) / 3.0, float(positive.max()) * 3.0)
    else:
        ax.set_ylim(*cosine_axis_limits(values))
    fig.tight_layout()
    return save_pdf(fig, filename)


def plot_all_diagnostics():
    task_key = TASK_CONFIG["task_key"]
    plot_loss_curves_10seed(history_df)
    if TASK_CONFIG["task_type"] == "classification":
        plot_score_curves_10seed(history_df)
    plot_diagnostic_bar(diagnostic_summary_df, "cosine_mean", "Cosine similarity", f"{task_key}_cosine_10seed.pdf", log_scale=False)
    plot_diagnostic_bar(diagnostic_summary_df, "variance_mean", "Estimator variance", f"{task_key}_variance_10seed.pdf", log_scale=True)
    for checkpoint_epoch in sorted(checkpoint_summary_df["checkpoint_epoch"].unique()):
        checkpoint_df = checkpoint_summary_df[checkpoint_summary_df["checkpoint_epoch"] == checkpoint_epoch]
        plot_diagnostic_bar(checkpoint_df, "cosine_mean", "Cosine similarity", f"{task_key}_cosine_checkpoint_{int(checkpoint_epoch):03d}_10seed.pdf", log_scale=False)
        plot_diagnostic_bar(checkpoint_df, "variance_mean", "Estimator variance", f"{task_key}_variance_checkpoint_{int(checkpoint_epoch):03d}_10seed.pdf", log_scale=True)

plot_all_diagnostics()


In [ ]:

SUMMARY_SIGNIFICANT_FIGURES = 3


def _format_scientific_sigfigs(value, significant_figures=SUMMARY_SIGNIFICANT_FIGURES):
    mantissa, exponent = f"{value:.{significant_figures - 1}e}".split("e")
    return f"{mantissa}e{int(exponent)}"


def format_summary_number(value):
    if pd.isna(value):
        return ""
    value = float(value)
    if value == 0.0:
        return "0.00"
    abs_value = abs(value)
    if abs_value < 1e-3 or abs_value >= 1e4:
        return _format_scientific_sigfigs(value)
    exponent = int(np.floor(np.log10(abs_value)))
    decimals = SUMMARY_SIGNIFICANT_FIGURES - exponent - 1
    rounded_value = round(value, decimals)
    rounded_abs = abs(rounded_value)
    if rounded_abs == 0.0:
        return "0.00"
    if rounded_abs < 1e-3 or rounded_abs >= 1e4:
        return _format_scientific_sigfigs(rounded_value)
    rounded_exponent = int(np.floor(np.log10(rounded_abs)))
    rounded_decimals = max(0, SUMMARY_SIGNIFICANT_FIGURES - rounded_exponent - 1)
    return f"{rounded_value:.{rounded_decimals}f}"


def format_pm(mean, std, is_epoch=False):
    if pd.isna(mean):
        return ""
    if is_epoch:
        if pd.isna(std):
            return str(int(round(float(mean))))
        return f"{int(round(float(mean)))} $\\pm$ {int(round(float(std)))}"
    if pd.isna(std):
        return format_summary_number(mean)
    return f"{format_summary_number(mean)} $\\pm$ {format_summary_number(std)}"


def make_formatted_table(table_summary_df):
    rows = []
    for method in TASK_CONFIG["methods"]:
        row = table_summary_df[table_summary_df["method"] == method].iloc[0]
        rows.append({
            "Task": row["task"],
            "Method": row["method_label"],
            "Best train loss": format_pm(row["best_train_loss_mean"], row["best_train_loss_std"]),
            "Best test loss": format_pm(row["best_test_loss_mean"], row["best_test_loss_std"]),
            "Best train score": format_pm(row["best_train_score_mean"], row["best_train_score_std"]),
            "Best test score": format_pm(row["best_test_score_mean"], row["best_test_score_std"]),
            "Score metric": row["score_metric"],
            "Convergence epoch": format_pm(row["convergence_epoch_mean"], row["convergence_epoch_std"], is_epoch=True),
            "Cosine": format_pm(row["cosine_mean"], row["cosine_std"]),
            "Variance": format_pm(row["variance_mean"], row["variance_std"]),
        })
    return pd.DataFrame(rows)

formatted_table = make_formatted_table(table_summary_df)
display(formatted_table)

task_key = TASK_CONFIG["task_key"]
history_df.to_csv(DATA_OUT_DIR / f"{task_key}_history_by_seed.csv", index=False)
summary_seed_df.to_csv(DATA_OUT_DIR / f"{task_key}_training_summary_by_seed.csv", index=False)
diagnostics_df.to_csv(DATA_OUT_DIR / f"{task_key}_diagnostics_by_seed_checkpoint_layer.csv", index=False)
checkpoint_seed_df.to_csv(DATA_OUT_DIR / f"{task_key}_diagnostics_by_seed_checkpoint.csv", index=False)
diagnostic_seed_summary_df.to_csv(DATA_OUT_DIR / f"{task_key}_diagnostics_by_seed_overall.csv", index=False)
diagnostic_summary_df.to_csv(DATA_OUT_DIR / f"{task_key}_diagnostics_summary_10seed.csv", index=False)
checkpoint_summary_df.to_csv(DATA_OUT_DIR / f"{task_key}_diagnostics_checkpoint_summary_10seed.csv", index=False)
table_seed_df.to_csv(TABLE_DIR / f"{task_key}_results_table_by_seed.csv", index=False)
table_summary_df.to_csv(TABLE_DIR / f"{task_key}_results_table_raw_10seed.csv", index=False)
formatted_table.to_csv(TABLE_DIR / f"{task_key}_results_table_10seed.csv", index=False)
formatted_table.to_latex(TABLE_DIR / f"{task_key}_results_table_10seed.tex", index=False, escape=False)

with open(DATA_OUT_DIR / f"{task_key}_10seed_outputs.pkl", "wb") as f:
    pickle.dump({
        "config": TASK_CONFIG,
        "history": history_df,
        "summary_seed": summary_seed_df,
        "diagnostics": diagnostics_df,
        "checkpoint_seed": checkpoint_seed_df,
        "diagnostic_seed_summary": diagnostic_seed_summary_df,
        "diagnostic_summary": diagnostic_summary_df,
        "checkpoint_summary": checkpoint_summary_df,
        "table_seed": table_seed_df,
        "table_summary": table_summary_df,
        "formatted_table": formatted_table,
    }, f)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"Saved outputs to: {OUTPUT_DIR}")
print(f"Zip archive: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Not running in Colab or automatic download unavailable.")
    print(exc)
